In [2]:
%pip install pandas openpyxl pyarrow matplotlib seaborn scikit-learn statsmodels

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd

df = pd.read_excel("../data/Modified_loan_dataset_investor.xlsx")
df.to_parquet("../data/loans.parquet")   # use this from now on

In [4]:
df.shape


(843755, 22)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 843755 entries, 0 to 843754
Data columns (total 22 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   loan_id                       843755 non-null  str           
 1   country                       843755 non-null  str           
 2   loan_issued_at                843755 non-null  datetime64[us]
 3   early_repaid_at               252308 non-null  object        
 4   issued_amount                 843755 non-null  float64       
 5   loan_status                   843755 non-null  str           
 6   initial_interest_rate         843755 non-null  float64       
 7   nr_of_payments                843755 non-null  int64         
 8   principal_balance             836042 non-null  float64       
 9   principal_debt                836042 non-null  float64       
 10  principal_paid_total          836042 non-null  float64       
 11  interest_paid_total     

In [6]:
df["loan_issued_at"].dt.year.value_counts().sort_index()       # To See how many loans each year has

loan_issued_at
2009       667
2010      1163
2011       451
2012       457
2013      2519
2014      8713
2015     14338
2016     10539
2017     17990
2018     25359
2019     56506
2020     27874
2021     51742
2022     63594
2023     81556
2024    147255
2025    182056
2026    150976
Name: count, dtype: int64

**Analysing data's from 2023 to 2025**

In [7]:
df = df[(df["loan_issued_at"] >= "2023-01-01") & (df["loan_issued_at"] < "2026-01-01")].copy()

print("First loan:", df["loan_issued_at"].min())
print("Last loan:", df["loan_issued_at"].max())
print("Rows and columns:", df.shape)

First loan: 2023-01-01 00:42:48
Last loan: 2025-12-31 23:26:15.567000
Rows and columns: (410867, 22)


In [8]:
# Recheck missing values in the filtered dataset from 2023 to 2025

missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(1)
})
missing[missing["missing_count"] > 0].sort_values("missing_pct", ascending=False)

,missing_count,missing_pct
months_in_default,354503,86.3
early_repaid_at,278774,67.9
next_payment_date_local,223713,54.4
customer_risk_rating,56467,13.7
has_default_within_12_months,37141,9.0
months_on_book,2487,0.6
combined_income,2,0.0


## Combined_income - Borrower's income, combined from bank statement and registry data (tax)

**The data dictionary says combined_income is “income combined from bank statement and registry data”. So a value only exists if Bondora actually collected one of those for that borrower**

**The missing rows are older loans (2014, 2019, 2020, 2021, 2022). Most likely Bondora didn’t collect or store income this way back then**

**From the year **2009 – 2021** no recording have been made for the combined_income (its 100% Missing). In the year **2022** - 86% of combined_income records are missing and followed by the year **2023 - 2025** - 0%. so that means there is no missing records for the combined_income**.

In [9]:

# Share of loans with missing income by country

df.groupby("country", observed=True)["combined_income"].apply(lambda s: s.isna().mean()).round(2)

country
Denmark        0.0
Estonia        0.0
Finland        0.0
Latvia         0.0
Lithuania      1.0
Netherlands    0.0
Spain          0.0
Name: combined_income, dtype: float64

## Followed by the combined_income as mentioned in the above markdown cell, from the **year 2022** they started to record the combined_income and rating system

**Countries where Bondora lent mainly in the early years show high missing shares. It doesn’t mean some countries hide income**

In [10]:
# Is income missing for the same loans as the rating?

pd.crosstab(df["combined_income"].isna(), df["customer_risk_rating"].isna())

customer_risk_rating,False,True
combined_income,,
False,354399,56466
True,1,1


## Income vs. rating: missing for the same loans

*(All loans, before the 2022–2025 filter)*

- 354,399 loans have both customer_risk_rating and Comboined_income · 273,153 have neither · 65,226 have income only · 1 has rating only.
- When income is missing, the rating is missing too, because both were not recorded for older loans.
- We cannot fill these values, so we keep them empty and analyse income and rating only for loans from **2022 onwards**.


**When income is missing, the rating is missing too. Both started being recorded around the same time**. 

**The Missing values cannot be filled from the data. We keep them as missing and analyse income and rating only for loans issued from 2022 onwards** 

In [11]:
df[df["combined_income"].isna()]    # We checked the loans with missing income and found that they are all missing the customer risk rating as well. So we can drop these rows.

,loan_id,country,loan_issued_at,early_repaid_at,issued_amount,loan_status,initial_interest_rate,nr_of_payments,principal_balance,principal_debt,...,is_default,next_payment_date_local,months_in_default,months_on_book,loan_status_risk,repaid_amount_total,initial_loan_duration,combined_income,has_default_within_12_months,customer_risk_rating
92090,0195DCF0-7A27-4564-998B-9CD178FA1EA9,Lithuania,2025-03-28 16:21:46.843,2026-05-22 13:12:02.050000,1000.0000,Repaid,0.1782,1,0.00000,0.00000,...,0.0,NaN,NaN,13.0,Written Off,1220.62000,60,NaN,0.0,NaN
382417,0195AA87-869E-41B9-8DD8-D3FBB91446AD,Denmark,2025-03-18 20:41:54.277,NaN,1380.6971,Active,0.1941,60,1113.22654,36.53753,...,0.0,2026-11-02 00:00:00,NaN,19.0,Active,622.23995,60,NaN,0.0,F


In [12]:
# Dropping the 2 rows in the combined_income column that are missing values.
# checking the number of rows removed and the number of missing values in the combined_income column after dropping the rows.
before = len(df)
df = df[df["combined_income"].notna()].copy()

print("Removed:", before - len(df))                                   # 2
print("Missing combined_income:", df["combined_income"].isna().sum()) # 0
print("Rows now:", len(df))

Removed: 2
Missing combined_income: 0
Rows now: 410865


In [13]:
# Share of loans with missing income by year

df.groupby(df["loan_issued_at"].dt.year)["combined_income"].apply(lambda s: s.isna().mean()).round(2)

loan_issued_at
2023    0.0
2024    0.0
2025    0.0
Name: combined_income, dtype: float64

***Analysing the unique values***

In [14]:
print(df["is_default"].unique())

[0. 1.]


In [15]:
print(df["has_default_within_12_months"].unique())

[ 0.  1. nan]


In [16]:
print(df["customer_risk_rating"].unique())

<ArrowStringArray>
[nan, 'D', 'AA', 'F', 'A', 'C', 'B', 'E', 'G', 'HR']
Length: 10, dtype: str


In [17]:
# Changing the data types of the columns to save memory and improve performance. 
# The "is_default" and "has_default_within_12_months" columns are converted to Int8, and the categorical columns are converted to category type.
df["is_default"] = df["is_default"].astype("Int8")
df["has_default_within_12_months"] = df["has_default_within_12_months"].astype("Int8")

for col in ["country", "loan_status", "loan_status_risk", "customer_risk_rating"]:
    df[col] = df[col].astype("category")

df.info()

<class 'pandas.DataFrame'>
Index: 410865 entries, 1 to 843753
Data columns (total 22 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   loan_id                       410865 non-null  str           
 1   country                       410865 non-null  category      
 2   loan_issued_at                410865 non-null  datetime64[us]
 3   early_repaid_at               132092 non-null  object        
 4   issued_amount                 410865 non-null  float64       
 5   loan_status                   410865 non-null  category      
 6   initial_interest_rate         410865 non-null  float64       
 7   nr_of_payments                410865 non-null  int64         
 8   principal_balance             410865 non-null  float64       
 9   principal_debt                410865 non-null  float64       
 10  principal_paid_total          410865 non-null  float64       
 11  interest_paid_total          

In [18]:
# Changing the order of the customer_risk_rating column to be in the order of riskiness, from least risky to most risky.

order = ["AA", "A", "B", "C", "D", "E", "F", "G", "HR"]  # replace with your actual values
df["customer_risk_rating"] = pd.Categorical(df["customer_risk_rating"], categories=order, ordered=True)

**customer_risk_rating grades each customer at the time the loan was issued. **AA is the best grade** (the most reliable customers, least likely to default) and **HR is the lowest grade** (the riskiest customers, most likely to default)**.

**We converted the column to an **ordered category**, from best to lowest grade:**

**AA → A → B → C → D → E → F → G → HR**

In [19]:
# Dropping the "nr_of_payments" column as it is not needed for analysis.

df = df.drop(columns=["nr_of_payments"])

**nr_of_payments was removed because it is only filled on the day a payment is due, so its value depends on the data extraction date rather than on the loan itself. It carries no information about default risk, and months_on_book already captures how far along each loan is**

In [20]:
df.shape

(410865, 21)

In [21]:
df["is_default"].isna().sum()     # counting the number of missing values in the column "is_default"

np.int64(0)

In [22]:
df["combined_income"].isna().sum()

np.int64(0)

In [23]:
df["customer_risk_rating"].isna().sum()


np.int64(56466)

In [24]:
df["has_default_within_12_months"].isna().sum()

np.int64(37141)

In [25]:
df["months_in_default"].isna().sum()

np.int64(354501)

**The total dataset is (410867), but when we check here at months_in_default, it shows that missing values are (354503). But none is not default. Blank means “not in default” rather than “unknown”. As per the dataset dictionary, the number of months since default occurred, NULL if not default. Amount of months in default.**

In [26]:
df.describe()

,loan_issued_at,issued_amount,initial_interest_rate,principal_balance,principal_debt,principal_paid_total,interest_paid_total,is_default,months_in_default,months_on_book,repaid_amount_total,initial_loan_duration,combined_income,has_default_within_12_months
count,410865,410865.000000,410865.000000,410865.000000,410865.000000,410865.000000,410865.000000,410865.0,56364.000000,408378.00000,410865.000000,410865.000000,410865.000000,373724.0
mean,2024-10-11 08:10:32.225828,1983.515837,0.182005,1018.423534,257.867982,924.761630,374.219509,0.137184,11.312167,15.37766,1346.862543,58.420450,2552.672156,0.086949
min,2023-01-01 00:42:48,10.000000,0.045000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.00000,0.000000,3.000000,0.000000,0.0
25%,2024-03-28 06:20:04.160000,425.000000,0.174800,0.000000,0.000000,103.000000,35.920000,0.0,5.000000,8.00000,222.070000,36.000000,1693.010000,0.0
50%,2024-11-14 20:42:56.213000,1036.000000,0.180500,195.180000,0.000000,348.430000,137.840000,0.0,10.000000,14.00000,621.890000,60.000000,2338.210000,0.0
75%,2025-06-19 08:17:23.543000,3109.000000,0.192700,1214.580000,0.000000,1036.000000,462.260000,0.0,17.000000,22.00000,1747.880000,78.000000,3040.790000,0.0
max,2025-12-31 23:26:15.567000,20000.000000,0.336500,19744.280000,19744.280000,20000.000000,10539.070000,1.0,42.000000,46.00000,23058.580000,120.000000,177871.700000,1.0
std,NaN,2305.590673,0.048268,1845.713784,978.322399,1524.050584,568.441073,0.344042,8.530062,10.30578,1819.672367,29.012322,1599.528709,0.281761


In [27]:
df.isna().mean().sort_values(ascending=False)   # missing values in Percentage in Descending order

months_in_default               0.862816
early_repaid_at                 0.678503
next_payment_date_local         0.544490
customer_risk_rating            0.137432
has_default_within_12_months    0.090397
months_on_book                  0.006053
loan_issued_at                  0.000000
country                         0.000000
loan_id                         0.000000
issued_amount                   0.000000
loan_status                     0.000000
principal_balance               0.000000
initial_interest_rate           0.000000
is_default                      0.000000
interest_paid_total             0.000000
principal_paid_total            0.000000
principal_debt                  0.000000
repaid_amount_total             0.000000
loan_status_risk                0.000000
combined_income                 0.000000
initial_loan_duration           0.000000
dtype: float64

In [28]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(1)
})
missing[missing["missing_count"] > 0].sort_values("missing_pct", ascending=False)

,missing_count,missing_pct
months_in_default,354501,86.3
early_repaid_at,278773,67.9
next_payment_date_local,223712,54.4
customer_risk_rating,56466,13.7
has_default_within_12_months,37141,9.0
months_on_book,2487,0.6


In [29]:
# Filling the missing values in the "months_in_default" column with 0 and converting the column to integer type.
df["months_in_default_flag"] = df["months_in_default"].fillna(0).astype(int)


In [30]:
print(df["months_in_default"].value_counts(dropna=False))    # Counting the values of the months_in_default column to see how many defaulted and non defaulted
print(df["months_in_default_flag"].value_counts()) 

months_in_default
NaN     354501
2.0       3120
6.0       3034
10.0      2927
3.0       2842
1.0       2773
5.0       2765
0.0       2716
7.0       2625
8.0       2559
9.0       2453
4.0       2363
11.0      2173
15.0      2170
12.0      2047
13.0      2004
19.0      1915
17.0      1909
14.0      1846
16.0      1835
18.0      1508
20.0      1114
22.0      1084
21.0      1066
23.0       831
24.0       596
27.0       420
26.0       372
25.0       350
31.0       326
34.0       284
32.0       269
29.0       268
28.0       239
30.0       225
36.0       224
35.0       220
33.0       219
37.0       194
38.0       184
39.0       154
40.0       100
41.0        36
42.0         5
Name: count, dtype: int64
months_in_default_flag
0     357217
2       3120
6       3034
10      2927
3       2842
1       2773
5       2765
7       2625
8       2559
9       2453
4       2363
11      2173
15      2170
12      2047
13      2004
19      1915
17      1909
14      1846
16      1835
18      1508
20      1114


**Handling `months_in_default`**

`months_in_default` counts how many months a loan has been in default. According to the data dictionary, an empty value means the loan has never defaulted (it was repaid or is still being paid) so these are not real missing values.

We did not fill the empty values with 0, because 0 already means "defaulted less than a month ago" (2,716 loans). Instead we created `months_in_default_flag`: 1 = the loan has defaulted, 0 = it has never defaulted. The original column is kept for analysing how long defaulted loans stay in default.

In [31]:
df["months_in_default_flag"] = df["months_in_default"].notna().astype(int)      # Checking how many defaulted and non defaulted in months_in_default column

df["months_in_default_flag"].value_counts()

months_in_default_flag
0    354501
1     56364
Name: count, dtype: int64

In [32]:
pd.crosstab(df["is_default"], df["months_in_default_flag"])  

months_in_default_flag,0,1
is_default,,
0,354501,0
1,0,56364


**Check: `is_default` vs `months_in_default_flag`**

The two flags agree for every loan: all 354,503 non-defaulted loans have no `months_in_default` value, and all 56,364 defaulted loans have one. This confirms that an empty `months_in_default` means the loan never defaulted. We use `is_default` for the analysis.

In [33]:
missing_12m = df[df["has_default_within_12_months"].isna()]

print(len(missing_12m))
print(missing_12m["loan_issued_at"].agg(["min", "max"]))   #Counts the min and max loan issued date for the missing_12m dataset to see if they are recent loans
print(missing_12m["loan_status"].value_counts())           #Counts for the loan_status column to see how many loans are in each status
print(missing_12m["is_default"].value_counts())            #Counts for the is_default column to see how many loans are defaulted and non-defaulted

37141
min   2025-10-24 00:13:37.173
max   2025-12-31 23:26:15.567
Name: loan_issued_at, dtype: datetime64[us]
loan_status
Active       25466
Repaid        9211
Defaulted     2464
Name: count, dtype: int64
is_default
0    34677
1     2464
Name: count, dtype: Int64


**Why `has_default_within_12_months` is empty**

37,141 loans have no value in `has_default_within_12_months`. All of them were issued between **24 October 2025 and 31 December 2025**, so they were less than 12 months old when the dataset was published (26 October 2026). This matches the data dictionary: the value is only filled once a loan has been on the books for 12 months.

**These loans split by status as follows:**

| Loan status | Loans | Is the 12-month outcome known? | Treatment |
|---|---|---|---|
| Defaulted | 2,464 | Yes, they defaulted within their first 12 months | Set to **1** |
| Repaid | 9,211 | Yes, a fully repaid loan can no longer default | Set to **0** |
| Active | 25,466 | No, they could still default before month 12 | Left **empty** |

**The 2,464 defaulted loans match exactly the loans with `is_default = 1` in this group, which confirms the statuses are consistent**.

**After filling, only the 25,466 active loans remain empty. The 12-month default rate is calculated on loans with a known outcome only**.

In [34]:
mask = df["has_default_within_12_months"].isna()

# Already defaulted → defaulted within 12 months

df.loc[mask & (df["is_default"] == 1), "has_default_within_12_months"] = 1

# Fully repaid → can no longer default

df.loc[mask & (df["loan_status"] == "Repaid"), "has_default_within_12_months"] = 0

# Check: what is still empty should be active loans only

print(df["has_default_within_12_months"].value_counts(dropna=False))
print(df[df["has_default_within_12_months"].isna()]["loan_status"].value_counts())

has_default_within_12_months
0       350440
1        34959
<NA>     25466
Name: count, dtype: Int64
loan_status
Active       25466
Defaulted        0
Repaid           0
Name: count, dtype: int64


In [35]:
df[df["has_default_within_12_months"].isna()]

,loan_id,country,loan_issued_at,early_repaid_at,issued_amount,loan_status,initial_interest_rate,principal_balance,principal_debt,principal_paid_total,...,next_payment_date_local,months_in_default,months_on_book,loan_status_risk,repaid_amount_total,initial_loan_duration,combined_income,has_default_within_12_months,customer_risk_rating,months_in_default_flag
36,008F634F-239E-46C1-98FD-B3A600C00F85,Finland,2025-12-01 11:47:13.503,NaN,2000.0000,Active,0.1748,1926.18000,0.00000,73.82000,...,2026-11-17 00:00:00,NaN,10.0,Active,439.51000,120,2636.29000,<NA>,A,0
58,00EA5C6D-509B-4847-B429-B3A90085E386,Finland,2025-12-04 12:02:15.280,NaN,11800.0000,Active,0.1748,11423.50000,0.00000,376.50000,...,2026-11-10 00:00:00,NaN,10.0,Active,2171.49000,120,2079.64000,<NA>,AA,0
131,019A25CD-4FCB-4981-991C-6A9EDFA327DC,Denmark,2025-10-27 15:19:49.083,NaN,690.3485,Active,0.1941,616.07641,8.04826,74.27212,...,2026-10-30 00:00:00,NaN,12.0,Active,188.18500,60,2678.28418,<NA>,A,0
134,019A45EF-710B-4A22-AFDD-C0DF0E3B2D34,Denmark,2025-11-02 21:15:29.543,NaN,897.4531,Active,0.1941,790.43700,0.00000,107.01609,...,2026-10-30 00:00:00,NaN,11.0,Active,263.76810,60,3695.71046,<NA>,A,0
135,019A46AC-6747-42F5-9339-676804C71985,Denmark,2025-11-03 00:38:52.330,NaN,1974.3968,Active,0.1941,1738.96515,0.00000,235.43164,...,2026-10-30 00:00:00,NaN,11.0,Active,580.30698,60,3684.04826,<NA>,B,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
843586,FC52EE35-0160-4CB5-8DA6-B39D00C2939C,Estonia,2025-11-22 11:54:37.150,NaN,318.0000,Active,0.1999,315.42000,0.00000,2.58000,...,46339,NaN,11.0,Active,63.66000,120,1319.02000,<NA>,AA,0
843609,FCBEB91D-86E7-4241-8E5D-B3B300E84A8C,Estonia,2025-12-14 14:08:44.227,NaN,212.0000,Active,0.2299,206.95000,0.00000,5.05000,...,46339,NaN,10.0,Active,52.19000,120,1971.11000,<NA>,A,0
843668,FDDE309C-0E20-4F8C-A18C-B39B0109CFF5,Finland,2025-11-20 16:52:13.777,NaN,1600.0000,Active,0.1748,1540.94000,0.00000,59.06000,...,46339,NaN,11.0,Active,358.25000,120,2082.33000,<NA>,B,0
843671,FDEC458A-9AF4-4660-B71A-B38E00FC167A,Finland,2025-11-07 19:41:18.230,NaN,200.0000,Active,0.1748,156.10000,0.52000,43.90000,...,46339,NaN,11.0,Active,71.60000,12,1745.85000,<NA>,C,0


**The borrowers who are active in the loan status and not has_defualt_within_12 months are considered as NA(Missing Values)**

In [36]:
df["has_default_within_12_months"].mean()  

np.float64(0.09070859031808594)

**It means 9.1% of loans defaulted within their first 12 months**.

In [37]:
# has_default_within_12_months is now filled for all loans that have already defaulted or have been fully repaid. 
# The remaining missing values are for active loans, which we cannot yet determine if they will default within 12 months. 
# Therefore, we will only analyze the loans that have a known value for this column "has_default_within_12_months".

df_12m = df[df["has_default_within_12_months"].notna()].copy()             

print(df_12m.shape)
print("12-month default rate:", round(df_12m["has_default_within_12_months"].mean(), 3))

(385399, 22)
12-month default rate: 0.091


In [38]:
df["is_default"].mean() 

np.float64(0.13718374648607207)

**That Means 13.7% of the loans are defaulted in the default_flag columns**

**"has_default_within_12months" rate**

We created `df_12m`, which keeps only the 385,401 loans whose 12-month outcome is known. The 25,466 active loans younger than 12 months are excluded, because their outcome is not yet known.

- Overall default rate: **13.7%** (56,364 of 410,867 loans)
- 12-month default rate: **9.1%** (of 385,401 loans)

The 12-month rate is lower because some loans only default after their first year. We use the 12-month rate to compare groups (rating, country, income), because every loan is observed for the same amount of time.

In [39]:
df[df["customer_risk_rating"].isna()]

,loan_id,country,loan_issued_at,early_repaid_at,issued_amount,loan_status,initial_interest_rate,principal_balance,principal_debt,principal_paid_total,...,next_payment_date_local,months_in_default,months_on_book,loan_status_risk,repaid_amount_total,initial_loan_duration,combined_income,has_default_within_12_months,customer_risk_rating,months_in_default_flag
1,0009E1E6-1880-44DE-9F3F-AFE9011E58C7,Finland,2023-04-19 18:17:39.000,NaN,4146.0,Repaid,0.1927,0.00,0.0,2438.49,...,NaN,NaN,15.0,Written Off,3010.50,60,2124.89,0,NaN,0
5,0014F40C-0920-4B6A-898B-AFC1001E4CE0,Finland,2023-03-10 01:54:53.000,2025-02-19 11:48:13.287000,4146.0,Repaid,0.1927,0.00,0.0,4146.00,...,NaN,NaN,23.0,Paid Up,5676.59,24,3211.36,0,NaN,0
23,005BAF23-5D08-415D-A3D7-AFAA00D85E52,Netherlands,2023-02-15 22:16:45.000,NaN,4040.0,Repaid,0.0920,0.00,0.0,1527.44,...,NaN,NaN,15.0,Written Off,1807.93,60,2255.42,0,NaN,0
42,00AF28C9-E833-47F2-9F78-B00700A33592,Finland,2023-05-19 13:06:06.000,2023-06-09 12:43:12,2073.0,Repaid,0.1927,0.00,0.0,2073.00,...,NaN,NaN,0.0,Paid Up,2092.76,18,1506.26,0,NaN,0
80,0156DC66-8795-4EB5-A1D0-AFBF00E1F165,Finland,2023-03-08 13:56:59.000,2023-03-09 12:44:17,518.0,Repaid,0.1927,0.00,0.0,518.00,...,NaN,NaN,0.0,Cancelled,518.28,6,3067.74,0,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
843633,FD3E8092-BEA0-48C2-8404-AFBF01437C73,Finland,2023-03-08 19:41:32.000,44993.999167,3627.0,Repaid,0.1927,0.00,0.0,3627.00,...,NaN,NaN,NaN,Cancelled,3627.00,60,2554.08,0,NaN,0
843685,FE51E2E1-78C4-4C87-AA42-B01B01055839,Finland,2023-06-08 15:52:36.000,NaN,621.0,Active,0.1927,541.66,0.0,79.34,...,46322,NaN,41.0,Active,504.30,119,1589.31,0,NaN,0
843706,FEEAA84A-E149-421F-8B8D-B00600FDFD30,Finland,2023-05-18 15:35:36.000,45300.698995,2487.0,Repaid,0.1927,0.00,0.0,2487.00,...,NaN,NaN,7.0,Paid Up,2779.95,60,1507.52,0,NaN,0
843713,FF29AD4C-2538-4292-8CC2-B046014B31C2,Estonia,2023-07-21 20:15:16.630,NaN,1382.0,Active,0.1899,682.54,0.0,699.46,...,46339,NaN,39.0,Active,1525.62,60,1942.18,0,NaN,0


***Missing customer risk rating***

56,467 loans have no risk rating. Unlike the older loans, these are recent (2023), they have income, and most are from Finland with the same interest rate (19.27%). This suggests a group of loans that Bondora issued without a rating.

**How we handle it**
- We **do not fill** the rating. It is Bondora's own risk assessment and cannot be rebuilt from other columns.
- We **do not delete** these loans. They have all other information and are still used in the analysis.
- We put them in a separate group, **"Unrated"**, so we can compare their default rate with the rated grades (AA to HR).

***These loans are not old. They’re from 2023, they have income, and many are from Finland with the same interest rate (19.27%). So the reason here is different from the older loans: Bondora probably issued a group of loans (maybe a specific product or campaign) without giving them a rating***.


***These loans still have a valid amount even though they are missing values because they have interest rate, country, income and outcome, so they’re useful for every other analysis***

In [40]:
#created an “Unrated” group for the missing values(keep the original column unchanged):
df["rating_group"] = (                             
    df["customer_risk_rating"]
    .cat.add_categories("Unrated")
    .fillna("Unrated")
)

df["rating_group"].value_counts()

rating_group
AA         93332
B          72408
C          59723
A          58692
Unrated    56466
D          35343
E          20268
F           8864
G           5750
HR            19
Name: count, dtype: int64

In [41]:
#comparing default rates, including unrated loans

df.groupby("rating_group", observed=True)["is_default"].agg(["count", "mean"])

,count,mean
rating_group,,
AA,93332,0.054611
A,58692,0.10136
B,72408,0.13667
C,59723,0.17181
D,35343,0.208245
E,20268,0.225429
F,8864,0.25519
G,5750,0.264696
HR,19,0.0


***Analysing Months_On_Book***

In [42]:
# Months on book is a column that indicates the number of months a loan has been active or not on the books.
# Months on book is a measure of the duration of a loan, which can be an important factor in assessing the risk of default.

mob = df[df["months_on_book"].isna()]
print(len(mob))
print(mob["loan_status"].value_counts())
print(mob["loan_status_risk"].value_counts())
print(mob["loan_issued_at"].dt.year.value_counts())



2487
loan_status
Repaid       2487
Active          0
Defaulted       0
Name: count, dtype: int64
loan_status_risk
Cancelled      2487
Active            0
Defaulted         0
Paid Up           0
Written Off       0
Name: count, dtype: int64
loan_issued_at
2024    1131
2025    1014
2023     342
Name: count, dtype: int64


**These 2,487 loans were cancelled.All 2,487 have loan_status = Repaid and loan_status_risk = Cancelled. The borrower took the loan and returned the money soon after, so the loan never really ran. Since there is no loan running, there are no months to count, which is why months_on_book is empty**

In [43]:
# loan_status_risk groups the loan status into risk categories: Low risk, Medium risk and High risk
# This risk categories split into Active, Paid Up, Defaulted, Cancelled and Written Off.
# Here we checked how many loans are in each group, where months_on_book is missing,
# and the default rate per group.

print(df["loan_status_risk"].value_counts())
print(pd.crosstab(df["loan_status_risk"], df["months_on_book"].isna()))
print(df.groupby("loan_status_risk", observed=True)["is_default"].mean().round(3))

loan_status_risk
Active         187123
Paid Up        136409
Defaulted       56364
Cancelled       16784
Written Off     14185
Name: count, dtype: int64
months_on_book     False  True 
loan_status_risk               
Active            187123      0
Cancelled          14297   2487
Defaulted          56364      0
Paid Up           136409      0
Written Off        14185      0
loan_status_risk
Active         0.0
Cancelled      0.0
Defaulted      1.0
Paid Up        0.0
Written Off    0.0
Name: is_default, dtype: Float64


***Written-off loans - the borrower paid back only part of the loan and then stopped paying. Bondora gave up trying to collect the rest and wrote it off as a loss.***

In [44]:
# Keeping cancelled loans and marking them with a flag

df["is_cancelled"] = (df["loan_status_risk"] == "Cancelled").astype(int)

print("Cancelled loans:", df["is_cancelled"].sum())                     # 16,784
print("Rows in df:", len(df))                                           # 410,865
print("Missing months_on_book:", df["months_on_book"].isna().sum())     # 2,487 (all cancelled)
print("Default rate:", round(df["is_default"].mean() * 100, 1), "%")  # 13.7%

Cancelled loans: 16784
Rows in df: 410865
Missing months_on_book: 2487
Default rate: 13.7 %


**Cancelled loans: kept in the dataset**

16,784 loans are marked as **Cancelled**. The borrower returned the money shortly after payout, so these loans never defaulted. After discussing with our coach, we **keep these loans** in the dataset and mark them with a new column, `is_cancelled` (1 = cancelled, 0 = not cancelled). This lets us include or exclude them in each analysis.

`months_on_book` is empty for 2,487 of these cancelled loans, because they were cancelled almost immediately and never ran. We keep these values as missing.

In [45]:
print("Default rate, all loans:", round(df["is_default"].mean() * 100, 1), "%")
print("Default rate, without cancelled:",
      round(df.loc[df["is_cancelled"] == 0, "is_default"].mean() * 100, 1), "%")

Default rate, all loans: 13.7 %
Default rate, without cancelled: 14.3 %


**Default rate with and without cancelled loans**

- **All loans:** 13.7% of loans defaulted.
- **Without cancelled loans:** 14.3% of loans defaulted.

The difference is small (0.6 percentage points). Cancelled loans can never default, so including them lowers the default rate slightly. We keep the cancelled loans in the dataset and use **13.7%** as our main default rate. The **14.3%** shows the default rate among loans that actually ran.

## Analysis for early_repaid_at and next_payment_date_local

In [46]:
# Is the value empty (True) or filled (False), per loan status?

print(pd.crosstab(df["loan_status"], df["early_repaid_at"].isna()))
print(pd.crosstab(df["loan_status"], df["next_payment_date_local"].isna()))

early_repaid_at   False   True 
loan_status                    
Active                0  187331
Defaulted             0   56363
Repaid           132092   35079
next_payment_date_local   False   True 
loan_status                            
Active                   187123     208
Defaulted                     0   56363
Repaid                       30  167141


**`early_repaid_at` and `next_payment_date_local`: missing by design**

- `early_repaid_at` is only filled for loans that were paid off early (132,092 repaid loans). Active and defaulted loans are always empty.
- `next_payment_date_local` is only filled for loans that still have a payment due (187,123 active loans). Repaid and defaulted loans are empty.
-  Small exceptions: 208 active loans have no next payment date and 30 repaid loans still have one, which is too few to matter.
- In the table above, **True** means the value is **empty** and **False** means a date **is filled in**.
-  We keep both columns as they are and do not fill them.

In [47]:
df[(df["loan_status"] == "Active") & df["next_payment_date_local"].isna()]["loan_status_risk"].value_counts()

loan_status_risk
Paid Up        208
Active           0
Cancelled        0
Defaulted        0
Written Off      0
Name: count, dtype: int64

**The 208 active loans without a next payment date**

All 208 loans that are "Active" in `loan_status` but have no next payment date are marked **"Paid Up"** in `loan_status_risk`. These borrowers have most likely just finished paying off their loan, and `loan_status` had not yet been updated when the data was extracted. That explains why there is no next payment date.

The two status columns disagree for only 208 loans (0.05%), which is too few to affect the analysis. We keep these loans unchanged.

In [48]:
def fix_excel_dates(s):
    num = pd.to_numeric(s, errors="coerce")                           # Excel numbers → numbers
    from_text = pd.to_datetime(s.where(num.isna()), errors="coerce", format="mixed")
    from_num = pd.to_datetime(num, unit="D", origin="1899-12-30")     # Excel day 0 = 30 Dec 1899
    return from_text.fillna(from_num)

for col in ["early_repaid_at", "next_payment_date_local"]:
    df[col] = fix_excel_dates(df[col])
    print(col, df[col].dtype, "| missing:", df[col].isna().sum())

early_repaid_at datetime64[ns] | missing: 278773
next_payment_date_local datetime64[us] | missing: 223712


**Missing by design: `early_repaid_at` and `next_payment_date_local`**

- `early_repaid_at` is empty for 278,773 loans (67.9%). It only has a value when a loan was paid off before its planned end date. Active and defaulted loans have not been repaid early, so empty means "no early repayment", not unknown.(Missing Value)
- `next_payment_date_local` is empty for 223,712 loans (54.4%). Repaid and defaulted loans have no next payment, so the field is empty.

Both columns contained a mix of dates and Excel date numbers (e.g. 46339), which we converted to proper dates. `next_payment_date_local` only describes the day the data was extracted, so we do not use it to explain default.

In [50]:
df.to_parquet("../data/loans_clean.parquet")
print("Saved:", df.shape)

Saved: (410865, 24)
